---
# Consistencia Semántica y Etiquetado de Tópicos con un LLM
## Python + Hugging Face
---

En la unidad anterior (**5a**) estimamos tres modelos LDA sobre el corpus de críticas de películas, con $k=12$, $k=20$ y $k=25$ tópicos, y exportamos las 15 palabras más representativas de cada tópico a tres archivos CSV.

En este notebook usaremos un **LLM de código abierto** (vía Hugging Face `transformers`) para dos tareas complementarias a lo que ya hicimos en R:

1. **Etiquetar** cada tópico con un nombre corto y descriptivo, a partir de sus palabras clave.
2. **Evaluar la consistencia semántica** (*topic coherence*) de cada tópico: ¿forman estas palabras un tema reconocible, o son una mezcla poco relacionada?

Esto nos da una segunda fuente de evidencia — cualitativa y semántica — para comparar los tres valores de $k$, complementando las métricas estadísticas ya vistas (perplexity, el método del codo).

## 🎯 Objetivos de aprendizaje

- Cargar un modelo instruccional open-source desde Hugging Face y ejecutarlo localmente o en Colab.
- Diseñar un *prompt* que le pida al LLM una salida en un formato específico y parseable.
- Aplicar el LLM sistemáticamente a los tópicos de los tres modelos ($k=12,20,25$).
- Comparar la coherencia semántica promedio entre valores de $k$ como evidencia adicional para elegir el número de tópicos.

## 🤖 ¿Qué modelo usamos?

Usaremos **`Qwen/Qwen2.5-7B-Instruct`**.

Es una alternativa a modelos como `meta-llama/Meta-Llama-3-8B-Instruct` con dos ventajas prácticas para este notebook:

- **No requiere acceso restringido ("gated")**: los pesos de Llama-3 en Hugging Face exigen aceptar una licencia y solicitar acceso (con aprobación no inmediata) antes de poder descargarlos, lo que agrega fricción a un ejercicio de curso. Qwen2.5 se descarga directamente.
- **Calidad comparable o superior** en tareas estructuradas como esta (leer una lista de palabras, producir una etiqueta y un puntaje), para un tamaño similar (~7-8B parámetros).

> 📌 El nombre del modelo está aislado en una sola variable (`MODEL_NAME`). Si prefieres usar Llama-3 (por ejemplo, porque ya tienes acceso aprobado y un token de Hugging Face configurado), basta con cambiar esa línea por `"meta-llama/Meta-Llama-3-8B-Instruct"` — el resto del notebook funciona igual.

> ⚠️ **GPU necesaria:** si estás en Google Colab, activa GPU en `Entorno de ejecución > Cambiar tipo de entorno de ejecución > GPU (T4)` antes de continuar. Cargamos el modelo en 4 bits (`bitsandbytes`) para que quepa cómodamente en la GPU gratuita de Colab.

## 📦 Instalación y carga de librerías

In [ ]:
!pip install -q transformers accelerate bitsandbytes torch pandas matplotlib

In [ ]:
import re

import pandas as pd
import matplotlib.pyplot as plt
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

## 🧠 Cargar el modelo

Cargamos el modelo cuantizado en 4 bits (`nf4`), lo que reduce significativamente el uso de memoria de GPU manteniendo una calidad de generación muy cercana a la versión completa.

In [ ]:
MODEL_NAME = "Qwen/Qwen2.5-7B-Instruct"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_quant_type="nf4",
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map="auto",
)

In [ ]:
def query_llm(prompt, max_new_tokens=200):
    """Envía un prompt al modelo usando su plantilla de chat y devuelve solo el texto generado."""
    messages = [{"role": "user", "content": prompt}]
    input_ids = tokenizer.apply_chat_template(
        messages, add_generation_prompt=True, return_tensors="pt"
    ).to(model.device)

    output = model.generate(
        input_ids,
        max_new_tokens=max_new_tokens,
        do_sample=False,
    )

    respuesta = tokenizer.decode(output[0][input_ids.shape[-1]:], skip_special_tokens=True)
    return respuesta.strip()

## 📂 Cargar los tópicos exportados desde R (Unidad 5a)

Los tres archivos deben estar en el mismo directorio de trabajo que este notebook (o sube los archivos a Colab antes de ejecutar esta celda).

In [ ]:
topterms_12 = pd.read_csv("lda_topterms_k12.csv")
topterms_20 = pd.read_csv("lda_topterms_k20.csv")
topterms_25 = pd.read_csv("lda_topterms_k25.csv")

topterms_12.head()

## ✂️ Palabras clave por tópico

Una función simple para obtener, para un modelo y un tópico dado, sus $n$ palabras con mayor probabilidad ($\beta$) — ya vienen ordenadas desde R, pero lo hacemos explícito aquí.

In [ ]:
def get_topic_words(df, topic_id, n=15):
    palabras = (
        df[df["topic"] == topic_id]
        .sort_values("beta", ascending=False)
        .head(n)["term"]
        .tolist()
    )
    return palabras

get_topic_words(topterms_20, topic_id=1)

## ✍️ Diseño del prompt

Le pedimos al modelo una salida en un **formato fijo y parseable** (tres líneas con etiquetas conocidas), en lugar de una respuesta libre — esto hace posible extraer los resultados automáticamente con una expresión regular, sin depender de que el modelo "converse" de forma predecible.

In [ ]:
PROMPT_TEMPLATE = """Estas son las {n} palabras más representativas de un tópico obtenido mediante \
Latent Dirichlet Allocation (LDA), aplicado a un corpus de críticas de películas:

{words}

Responde EXACTAMENTE en este formato, sin texto adicional ni explicaciones fuera de él:

Etiqueta: <2 a 4 palabras que describan el tema común>
Coherencia (1-5): <número entero, donde 1 = las palabras no tienen relación temática entre sí y 5 = forman un tema claro y coherente>
Justificación: <una frase breve explicando tu evaluación de coherencia>
"""


def build_prompt(words):
    return PROMPT_TEMPLATE.format(n=len(words), words=", ".join(words))


print(build_prompt(get_topic_words(topterms_20, topic_id=1)))

## 🔍 Extraer la etiqueta, el puntaje y la justificación de la respuesta

In [ ]:
def parse_response(response):
    etiqueta = re.search(r"Etiqueta:\s*(.+)", response)
    coherencia = re.search(r"Coherencia \(1-5\):\s*(\d)", response)
    justificacion = re.search(r"Justificaci[oó]n:\s*(.+)", response)

    return {
        "etiqueta": etiqueta.group(1).strip() if etiqueta else None,
        "coherencia": int(coherencia.group(1)) if coherencia else None,
        "justificacion": justificacion.group(1).strip() if justificacion else None,
    }

## 🔁 Aplicar el LLM a todos los tópicos de un modelo

Recorremos cada tópico del modelo, construimos su prompt, consultamos al LLM y guardamos el resultado parseado en un `DataFrame`.

> ⏱️ Esta celda puede tardar varios minutos por modelo, ya que genera una respuesta por cada tópico (12, 20 o 25 llamadas al modelo).

In [ ]:
def evaluate_model_topics(df, k, n_words=15):
    resultados = []
    for topic_id in sorted(df["topic"].unique()):
        palabras = get_topic_words(df, topic_id, n=n_words)
        prompt = build_prompt(palabras)
        respuesta = query_llm(prompt)
        parsed = parse_response(respuesta)
        parsed.update({"k": k, "topic": topic_id, "palabras": ", ".join(palabras)})
        resultados.append(parsed)

    return pd.DataFrame(resultados)

In [ ]:
resultados_12 = evaluate_model_topics(topterms_12, k=12)
resultados_20 = evaluate_model_topics(topterms_20, k=20)
resultados_25 = evaluate_model_topics(topterms_25, k=25)

resultados_todos = pd.concat([resultados_12, resultados_20, resultados_25], ignore_index=True)
resultados_todos

## 📊 Comparar la coherencia semántica promedio por valor de $k$

Promediamos el puntaje de coherencia asignado por el LLM dentro de cada modelo, para tener un resumen comparable con las métricas estadísticas (perplexity, método del codo) obtenidas en R.

In [ ]:
coherencia_promedio = resultados_todos.groupby("k")["coherencia"].mean()
coherencia_promedio

In [ ]:
plt.figure(figsize=(6, 4))
coherencia_promedio.plot(kind="bar", color=["#175895", "#175895", "#175895"])
plt.ylabel("Coherencia promedio (1-5, según el LLM)")
plt.xlabel("Número de tópicos (k)")
plt.title("Coherencia semántica promedio por valor de k")
plt.ylim(0, 5)
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

## 🏷️ Revisar las etiquetas generadas

Una forma rápida de inspeccionar cualitativamente los resultados: ver las etiquetas y palabras clave de un modelo en particular.

In [ ]:
resultados_20[["topic", "etiqueta", "coherencia", "palabras"]]

## ✅ Cierre

Este notebook agrega una evidencia **cualitativa y semántica** a la selección de $k$, complementando las métricas puramente estadísticas de la Unidad 5a:

- **R / Quanteda + topicmodels**: produce los tópicos y las métricas de ajuste (perplexity) de forma reproducible y computacionalmente eficiente.
- **Python + LLM**: interpreta el *contenido* de esos tópicos — algo que ninguna métrica estadística puede hacer por sí sola — y propone etiquetas listas para presentar a una audiencia no técnica.

> 📌 **Idea clave:** ni el enfoque estadístico ni el LLM son suficientes por separado para elegir y comunicar un buen modelo de tópicos — la combinación de ambos es lo que da una respuesta completa.